# task_senegal — baseline walkthrough

Loads the 20-question fact bank and the intern's naive starting config
(`dataset/example_naive_config.json` — every question routed to the large
tier, verbose style, few-shot attached), and runs it through the shared
`model_utils.run_config` to show the symptom: it clears the quality bar on
every question, but overshoots the token budget by a wide margin. This
notebook does **not** implement a fix or reveal the flag — it demonstrates
the I/O contract and why the naive config is not a valid submission on its
own.

In [1]:
import model_utils as mu

fact_bank = mu.load_fact_bank()
naive_config = mu.load_json("dataset/example_naive_config.json")
print(f"{len(fact_bank)} questions in the battery")
print(f"token budget: {mu.TOKEN_BUDGET}")

20 questions in the battery
token budget: 1200


## The intern's starting point — every question to the largest model, verbose, with a few-shot block

In [2]:
result = mu.run_config(naive_config, fact_bank)
n_pass = sum(1 for info in result["per_question"].values() if info["pass"])
print(f"quality: {n_pass}/{len(fact_bank)} questions cleared their bar -> "
      f"{'PASS' if result['quality_gate'] else 'FAIL'}")
print(f"tokens: {result['total_tokens']} / budget {mu.TOKEN_BUDGET} -> "
      f"{'PASS' if result['budget_gate'] else 'FAIL'}")

quality: 20/20 questions cleared their bar -> PASS
tokens: 3626 / budget 1200 -> FAIL


## A peek at one answer

Every required fact is present (that's why quality is a clean 20/20) — the
cost is in the boilerplate around it, not the facts themselves.

In [3]:
print(result["per_question"]["q01"]["answer"])
print("tokens for q01:", result["per_question"]["q01"]["tokens"])

Thanks so much for reaching out to the Teranga FM helpline! Dakar; Atlantic Ocean. We hope this fully answers your question, feel free to call back any time with more!
tokens for q01: 180


## The symptom

The naive config answers every question correctly, but at roughly 2.6x the
token budget. Your job is to produce a `routing_config.json` that keeps
every question's quality bar cleared while bringing the total under
`model_utils.TOKEN_BUDGET` — see `task_description.md` for the exact
contract and `main_template.py` for the I/O shape your submission must
match. `model_utils.py` is fully readable; nothing about the cost or
quality mechanism is hidden from you.

## I/O contract — what your submission must do

In [4]:
# main_template.py defaults to reading routing_config.json (the file you are
# expected to author), which does not exist yet in this demo, so pass
# --config explicitly to try the naive starting point and see the I/O
# contract work end to end: read the battery and a config by filename only,
# write answers by filename only.
import subprocess
result_run = subprocess.run(
    ["python", "main_template.py", "--config", "dataset/example_naive_config.json",
     "--output", "demo_answers.json"],
    capture_output=True, text=True,
)
print(result_run.stdout)
print(result_run.stderr)

wrote 20 answers to demo_answers.json
quality: 20/20 questions cleared their bar -> PASS
tokens: 3626 / budget 1200 -> FAIL




In [5]:
answers = mu.load_json("demo_answers.json")
print(answers[:3])
import os
os.remove("demo_answers.json")

[{'id': 'q01', 'answer': 'Thanks so much for reaching out to the Teranga FM helpline! Dakar; Atlantic Ocean. We hope this fully answers your question, feel free to call back any time with more!'}, {'id': 'q02', 'answer': 'Thanks so much for reaching out to the Teranga FM helpline! CFA franc; West African Economic and Monetary Union. We hope this fully answers your question, feel free to call back any time with more!'}, {'id': 'q03', 'answer': 'Thanks so much for reaching out to the Teranga FM helpline! French; Wolof. We hope this fully answers your question, feel free to call back any time with more!'}]
